<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">یک آموزش کوچک، با شاهد قابل دیدن</h1><p style="text-align:right"><b>پرسش آزمایش:</b> چگونه تغییر وزن، منحنی <bdi dir="ltr">Loss</bdi> و نمونهٔ تولیدشده را کنار هم بررسی کنیم؟</p><p style="text-align:right">پیش‌نیاز: <a target="_self" href="http://127.0.0.1:8000/part-07/chapter-01/43-lm-head.html"><bdi dir="ltr">43-lm-head</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-07/chapter-02/45-trace.html"><bdi dir="ltr">45-trace</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-07/chapter-02/46-gradient-path.html"><bdi dir="ltr">46-gradient-path</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-08/chapter-01/47-loop.html"><bdi dir="ltr">47-loop</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-08/chapter-01/48-evaluate.html"><bdi dir="ltr">48-evaluate</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-08/chapter-03/52-first-run.html"><bdi dir="ltr">52-first-run</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-08/chapter-03/53-curves.html"><bdi dir="ltr">53-curves</bdi></a></p><p style="text-align:right">این دفتر مستقل است و به اجرای دفتر دیگری نیاز ندارد. از بالا به پایین اجرا کنید؛ برای اجرای دوباره از ابتدا، <bdi dir="ltr">Kernel</bdi> را <bdi dir="ltr">Restart</bdi> و سپس <bdi dir="ltr">Run All</bdi> کنید. برای بازکردن لینک درس‌ها، سرور کتاب باید روی پورت ۸۰۰۰ اجرا شده باشد؛ راهنمای نصب در <a href="../../docs/NOTEBOOKS.md"><bdi dir="ltr">docs/NOTEBOOKS.md</bdi></a> است.</p>
</div>

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "mini_gpt" / "model.py").is_file()
             and (p / "data" / "sample.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("Keep notebooks inside the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Python:", sys.executable)
print("Project:", ROOT)

import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(17)

def inspect(name, value):
    print(name, "shape =", tuple(value.shape),
          "dtype =", value.dtype, "device =", value.device)


<div dir="rtl" style="text-align:right">
<p style="text-align:right">این دفتر از <bdi dir="ltr">MiniGPT</bdi>، <bdi dir="ltr">Tokenizer</bdi>، <bdi dir="ltr">Dataset</bdi>، <bdi dir="ltr">random_batch</bdi> و <bdi dir="ltr">evaluate</bdi> واقعی پروژه استفاده می‌کند. حلقهٔ آموزش در خود دفتر آمده تا بتوانید هر گام را دنبال کنید؛ برای ذخیره و <bdi dir="ltr">Resume</bdi> از فرمان‌های کتاب استفاده کنید. به فایل مدل از پیش ساخته‌شده نیاز ندارید و این دفتر چیزی در <bdi dir="ltr">runs</bdi> نمی‌نویسد. <bdi dir="ltr">Validation</bdi> انتهای جداشدهٔ همین پیکرهٔ کوچک است، نه آزمون تعمیم مستقل.</p>
</div>

In [ ]:
from torch.utils.data import DataLoader
from mini_gpt.config import ModelConfig
from mini_gpt.data import prepare_corpus
from mini_gpt.dataset import NextTokenDataset
from mini_gpt.model import MiniGPT
from mini_gpt.train import random_batch
from mini_gpt.evaluate import evaluate
train_ids,valid_ids,tokenizer,metadata = prepare_corpus(ROOT/"data"/"sample.txt",train_fraction=0.8)
config = ModelConfig(vocab_size=tokenizer.vocab_size,context_length=16,
                     embedding_dim=32,num_heads=4,num_layers=1,dropout=0.1)
model = MiniGPT(config).cpu()
optimizer = torch.optim.AdamW(model.parameters(),lr=0.003,weight_decay=0.01)
train_data = NextTokenDataset(train_ids,config.context_length)
valid_data = NextTokenDataset(valid_ids,config.context_length)
batch_rng = torch.Generator().manual_seed(18)
prompt_text = "مدل "
prompt = torch.tensor([tokenizer.encode(prompt_text)],dtype=torch.long)
assert 0 not in prompt[0].tolist()
print("Vocabulary:",list(enumerate(tokenizer.id_to_token)))
print("Parameters:",sum(p.numel() for p in model.parameters()))
print("Train/validation windows:",len(train_data),len(valid_data))
print("Validation unknown rate:",metadata["validation_unknown_rate"])


<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از یادگیری، مسیر را ببینیم</h2><p style="text-align:right">برای <bdi dir="ltr">B</bdi>=8 و <bdi dir="ltr">T</bdi>=16، شکل ورودی، <bdi dir="ltr">Target</bdi>، نمایش و <bdi dir="ltr">Logits</bdi> را پیش‌بینی کنید. <bdi dir="ltr">V</bdi> اندازهٔ <bdi dir="ltr">Vocabulary</bdi> است. آخرین <bdi dir="ltr">Target</bdi> هر پنجره، یک کاراکتر جلوتر از آخرین ورودی آن است.</p>
</div>

In [ ]:
x,y = random_batch(train_data,batch_size=8,generator=batch_rng)
model.eval()
trace = {}
with torch.no_grad():
    logits,loss = model(x,y,trace=trace)
for name,value in [("input IDs",x),("targets",y),
                   ("Token Embedding",trace["token_embedding"]),
                   ("Position Embedding",trace["position_embedding"]),
                   ("combined",trace["combined_embedding"]),
                   ("Q",trace["layers"][0]["attention"]["q"]),
                   ("mask",trace["layers"][0]["attention"]["mask"]),
                   ("weights",trace["layers"][0]["attention"]["weights"]),
                   ("block output",trace["layers"][0]["output"]),("logits",logits)]:
    inspect(name,value)
print("Input:",tokenizer.decode(x[0].tolist()))
print("Target:",tokenizer.decode(y[0].tolist()))
assert torch.equal(x[:,1:],y[:,:-1])
print("Initial Loss:",loss.item())


<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">آموزش و سنجش را جدا نگه داریم</h2><p style="text-align:right">مقدارهای هر دو منحنی روی کل <bdi dir="ltr">Split</bdi> مربوط و در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">eval</code> سنجیده می‌شوند: یک‌بار پیش از آموزش و سپس پس از گام‌های تعیین‌شدهٔ به‌روزرسانی. با <bdi dir="ltr">Loss</bdi> یک <bdi dir="ltr">Batch</bdi> در حالت <bdi dir="ltr">train</bdi> یکی نیستند. <bdi dir="ltr">Sampling</bdi> پنجره‌ها با جایگذاری است؛ تعداد <bdi dir="ltr">Step</bdi> را <bdi dir="ltr">Epoch</bdi> ننامید. اول پیش‌بینی کنید کاهش <bdi dir="ltr">Train Loss</bdi> چه چیزهایی را دربارهٔ <bdi dir="ltr">Validation</bdi> و متن تولیدی تضمین نمی‌کند.</p>
</div>

In [ ]:
train_loader = DataLoader(train_data,batch_size=64,shuffle=False,num_workers=0,
                          generator=torch.Generator().manual_seed(101))
valid_loader = DataLoader(valid_data,batch_size=64,shuffle=False,num_workers=0,
                          generator=torch.Generator().manual_seed(102))
history, samples = [], {}
def observe(step):
    train_loss = evaluate(model,train_loader,"cpu")
    valid_loss = evaluate(model,valid_loader,"cpu")
    history.append((step,train_loss,valid_loss))
    print(f"step={step:3d} train={train_loss:.3f} validation={valid_loss:.3f}")
    if step in (0,40,80,120):
        generated = model.generate(prompt,max_new_tokens=48,greedy=True)
        samples[step] = tokenizer.decode(generated[0].tolist())

observe(0)
for step in range(1,121):
    model.train()
    x,y = random_batch(train_data,batch_size=8,generator=batch_rng)
    optimizer.zero_grad(set_to_none=True)
    _,loss = model(x,y)
    if not torch.isfinite(loss):
        raise RuntimeError(f"Nonfinite Loss at step {step}")
    loss.backward()
    norm = torch.nn.utils.clip_grad_norm_(model.parameters(),1.0,error_if_nonfinite=True)
    if step == 1:
        parameter = model.token_embedding.weight
        before = parameter.detach().clone()
        inspect("Embedding Gradient",parameter.grad)
        print("Gradient norm before clipping:",norm.item())
    optimizer.step()
    if step == 1:
        print("Maximum embedding update:",(parameter.detach()-before).abs().max().item())
        assert not torch.equal(before,parameter.detach())
    if step % 20 == 0:
        observe(step)
for step,text in samples.items():
    print("\nStep",step)
    print(text)


In [ ]:
steps,train_losses,valid_losses = zip(*history)
fig,ax = plt.subplots(figsize=(7,4))
ax.plot(steps,train_losses,"o-",label="Train / eval mode")
ax.plot(steps,valid_losses,"o-",label="Validation / eval mode")
ax.set(xlabel="Optimizer steps",ylabel="Mean next-token Loss")
ax.legend()
ax.grid(alpha=0.25)
plt.show()


<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">همان مدل آموزش‌دیده را بازرسی کنیم</h2><p style="text-align:right">ابزار <bdi dir="ltr">inspect</bdi> عددها را از مدل همین <bdi dir="ltr">Kernel</bdi> می‌گیرد؛ تصویر نمونه یا خروجی ساختگی نیست. هر سطرِ نقشه <bdi dir="ltr">Query</bdi> و هر ستون <bdi dir="ltr">Key</bdi> است. چرا بالای قطر باید صفر بماند؟</p>
</div>

In [ ]:
from mini_gpt.inspect import inspect_model
report = inspect_model(model,tokenizer,prompt_text,layer=0,head=0,
                       max_tokens=config.context_length,generate_tokens=0,greedy=True)
print("Actual shapes:",report["shapes"])
print("Position to token:",list(enumerate(report["input"]["context_tokens"])))
print("Top candidates:",report["output"]["top_candidates"][:5])
weights = torch.tensor(report["attention"]["weights"])
torch.testing.assert_close(weights.sum(-1),torch.ones(weights.shape[0]))
assert torch.count_nonzero(weights.triu(1)) == 0
fig,ax = plt.subplots(figsize=(5,4))
plot = ax.imshow(weights,vmin=0,vmax=1,cmap="Blues")
ax.set(xlabel="Key position",ylabel="Query position",title="Trained layer 0 / head 0")
fig.colorbar(plot,ax=ax,label="Attention weight")
plt.show()


<div dir="rtl" style="text-align:right">
<p style="text-align:right"><b>آزمایش تغییریافته:</b> یک‌بار <bdi dir="ltr">Learning Rate</bdi> را از ۰٫۰۰۳ به ۰٫۳ ببرید و <bdi dir="ltr">Kernel</bdi> را از نو اجرا کنید. منحنی و نمونه‌ها را مقایسه کنید. چرا <bdi dir="ltr">Gradient</bdi> گام اول ثابت مانده، اما اندازهٔ تغییر وزن و مسیر <bdi dir="ltr">Loss</bdi> عوض شده‌اند؟ افت کیفیت یا نوسان ممکن است رخ دهد، ولی واگراییِ قطعی را برای هر مدل ادعا نکنید. سپس نرخ را برگردانید و فقط <bdi dir="ltr">context_length</bdi> را از ۱۶ به ۸ تغییر دهید. هر بار فقط یک عامل را عوض کنید.</p><p style="text-align:right"><b>برداشت:</b> افت <bdi dir="ltr">Loss</bdi> روی این دادهٔ کوچک، هوشمندی یا پاسخ‌گویی عمومی را ثابت نمی‌کند. نقشهٔ <bdi dir="ltr">Attention</bdi> هم ضرایب یک محاسبه است، نه توضیح کامل علت تولید.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">برگشت به کتاب</h2><p style="text-align:right">پیش‌بینی و نتیجهٔ اجرا را کنار هم بنویسید؛ اگر تفاوتی داشتند، دلیلش را توضیح دهید. سپس به <a target="_self" href="http://127.0.0.1:8000/part-08/chapter-03/53-curves.html">درس مرتبط</a> برگردید و نتیجه را با توضیح آن مقایسه کنید.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">تمرین تکمیلی: بعد از <bdi dir="ltr">Step</bdi> دقیقاً کدام وزن‌ها تغییر کردند؟</h2>
<p style="text-align:right">یک گزارش تغییر <bdi dir="ltr">Parameter</bdi> بسازید و <bdi dir="ltr">Snapshot</bdi> واقعی را از نمای مشترک حافظه جدا کنید. پیش‌نیاز: حلقهٔ آموزش و مقایسهٔ قبل/بعد از <bdi dir="ltr">Optimizer Step</bdi> همین دفتر را دیده‌اید. مثال‌های قبلی این دفتر را نگه داشته‌ایم. اکنون دو تابع <bdi dir="ltr">TODO</bdi> را خودتان بنویسید؛ <bdi dir="ltr">INCOMPLETE</bdi> یعنی کار هنوز تمام نشده است.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">آیا <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">parameter.detach()</code> به‌تنهایی عکس ثابتی از وزن می‌سازد؟ اگر وزن درجا عوض شود، آن <bdi dir="ltr">Tensor</bdi> چه می‌بیند؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
config = ModelConfig(9,6,8,2,1,0.)
model = MiniGPT(config).eval()
ids,targets = torch.tensor([[1,2,3],[2,3,4]]),torch.tensor([[2,3,4],[3,4,5]])
initial_state = {name:value.detach().clone() for name,value in model.state_dict().items()}
before = {name:p.detach().clone() for name,p in model.named_parameters()}
optimizer = torch.optim.SGD(model.parameters(),lr=0.01)
optimizer.zero_grad(set_to_none=True)
model(ids,targets)[1].backward(); optimizer.step()
after = {name:p.detach().clone() for name,p in model.named_parameters()}
print("one controlled SGD step; no additional training run")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">parameter_changes(before,after)</code> دیکشنری نام <bdi dir="ltr">Parameter</bdi> به <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">float</code> اندازهٔ تغییر آن برگرداند؛ اندازه ریشهٔ مجموع مربع اختلاف همهٔ مؤلفه‌هاست. ورودی‌ها <bdi dir="ltr">Snapshot</bdi>های هم‌کلیدند و نباید تغییر کنند.</p>
</div>

In [ ]:
def parameter_changes(before, after):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = parameter_changes(before,after)
    if result is None: return False
    assert set(result) == set(before)
    for name in before:
        assert math.isclose(result[name],(after[name]-before[name]).norm().item(),abs_tol=1e-8)
    assert result['language_model_head.weight'] > 0
    a = {'matrix':torch.zeros(2,2),'same':torch.ones(1)}
    b = {'matrix':torch.tensor([[3.,0.],[0.,4.]]),'same':torch.ones(1)}
    assert parameter_changes(a,b) == {'matrix':5.,'same':0.}
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط <bdi dir="ltr">Learning Rate</bdi> یک گام <bdi dir="ltr">SGD</bdi> را از صفر به ۰٫۰۱ و ۰٫۰۲ تغییر دهید. هر اجرا از همان <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">initial_state</code>، همان <bdi dir="ltr">Batch</bdi> و <bdi dir="ltr">Dropout</bdi> صفر آغاز شود.</p>
</div>

In [ ]:
for rate in (0.,0.01,0.02):
    candidate = MiniGPT(config).eval(); candidate.load_state_dict(initial_state)
    original = candidate.language_model_head.weight.detach().clone()
    optimizer = torch.optim.SGD(candidate.parameters(),lr=rate)
    candidate(ids,targets)[1].backward(); optimizer.step()
    print('rate, head update norm:',rate,(candidate.language_model_head.weight-original).norm().item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right"><bdi dir="ltr">Snapshot</bdi> خراب فقط <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">detach</code> شده و حافظه را با <bdi dir="ltr">Parameter</bdi> شریک است. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">snapshot_parameters(model)</code> برای هر <bdi dir="ltr">Parameter</bdi> مدل یک نسخهٔ مستقل و جدا از <bdi dir="ltr">Graph</bdi> بسازد.</p>
</div>

In [ ]:
wrong_snapshot = model.token_embedding.weight.detach()
with torch.no_grad(): model.token_embedding.weight.add_(0.1)
print('wrong reported change:',(model.token_embedding.weight-wrong_snapshot).abs().max().item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def snapshot_parameters(model):
    # TODO
    return None

In [ ]:
def test_repair():
    result = snapshot_parameters(model)
    if result is None: return False
    assert set(result) == {name for name,_ in model.named_parameters()}
    for name,p in model.named_parameters():
        assert not result[name].requires_grad
        assert result[name].data_ptr() != p.data_ptr()
        assert torch.equal(result[name],p)
    saved = result['token_embedding.weight'].clone()
    with torch.no_grad(): model.token_embedding.weight.add_(0.2)
    assert torch.equal(result['token_embedding.weight'],saved)
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right">گزارش از <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">MiniGPT</code> واقعی و یک <bdi dir="ltr">Step</bdi> واقعی گرفته شد. تغییر وزن شاهد اجرای به‌روزرسانی است، ولی به‌تنهایی کاهش <bdi dir="ltr">Validation Loss</bdi> یا بهترشدن زبان را ثابت نمی‌کند.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">چرا گزارش «هیچ وزنی تغییر نکرد» ممکن است خطای ابزار اندازه‌گیری باشد، نه خطای <bdi dir="ltr">Optimizer</bdi>؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-08/chapter-03/53-curves.html">بازگشت به درس مرتبط</a> · <a target="_self" href="http://127.0.0.1:8000/answers/lab-11_train_inspect.html">فقط پس از تلاش: پاسخ مرجع تمرین تکمیلی</a></p></div>